<a href="https://colab.research.google.com/github/ArmandoZaid/Final-Project-Hacktiv8-AI-For-Data-Scientist-/blob/main/Armando_Diaz_Final_Project_AI_Agents_AVPN_IT_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agents with LangGraph Part 1

In [ ]:
%pip install -qU langgraph langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.3/234.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.2.15 requires langgraph<1.2.0,>=1.1.5, but you have langgraph 1.2.0 which is incompatible.


## Exercise 1: SQL Agents

SQL agents adalah AI Agent yang dapat mengambil data dari table SQL dan menganalisis sesuai dengan pertanyaan dari user.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

from google.colab import userdata
GEMINI = userdata.get('GEMINI')
os.environ["GOOGLE_API_KEY"] = GEMINI

### Langkah 1: Define beberapa table SQL

In [ ]:
%load_ext sql
%sql sqlite:///sample.db

In [ ]:
%%sql
-- Create the 'products' table
CREATE TABLE IF NOT EXISTS products (
  	product_id INTEGER PRIMARY KEY AUTOINCREMENT,
  	product_name VARCHAR(255) NOT NULL,
  	price DECIMAL(10, 2) NOT NULL
  );

-- Create the 'staff' table
CREATE TABLE IF NOT EXISTS staff (
  	staff_id INTEGER PRIMARY KEY AUTOINCREMENT,
  	first_name VARCHAR(255) NOT NULL,
  	last_name VARCHAR(255) NOT NULL
  );

-- Create the 'orders' table
CREATE TABLE IF NOT EXISTS orders (
  	order_id INTEGER PRIMARY KEY AUTOINCREMENT,
  	customer_name VARCHAR(255) NOT NULL,
  	staff_id INTEGER NOT NULL,
  	product_id INTEGER NOT NULL,
  	FOREIGN KEY (staff_id) REFERENCES staff (staff_id),
  	FOREIGN KEY (product_id) REFERENCES products (product_id)
  );

-- Insert data into the 'products' table
INSERT INTO products (product_name, price) VALUES
  	('Laptop', 799.99),
  	('Keyboard', 129.99),
  	('Mouse', 29.99);

-- Insert data into the 'staff' table
INSERT INTO staff (first_name, last_name) VALUES
  	('Alice', 'Smith'),
  	('Bob', 'Johnson'),
  	('Charlie', 'Williams');

-- Insert data into the 'orders' table
INSERT INTO orders (customer_name, staff_id, product_id) VALUES
  	('David Lee', 1, 1),
  	('Emily Chen', 2, 2),
  	('Frank Brown', 1, 3);

 * sqlite:///sample.db
Done.
Done.
Done.
3 rows affected.
3 rows affected.
3 rows affected.


[]

In [ ]:
import sqlite3

db_file = "sample.db"

### Langkah 2: Define Tools
Tools yang akan digunakan pada SQL agent adalah `list_tables`, `describe_tables`, dan `execute_query` sehingga AI dapat memahami metadata table terlebih dahulu sebelum mulai menjalankan query

In [ ]:
def list_tables() -> list[str]:
    """Retrieve the names of all tables in the database."""
    # Include print logging statements so you can see when functions are being called.
    print(' - DB CALL: list_tables')

    # Create a new connection for thread safety
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()

        # Fetch the table names.
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

        tables = cursor.fetchall()
        return [t[0] for t in tables]


list_tables()

 - DB CALL: list_tables


['products', 'sqlite_sequence', 'staff', 'orders']

In [ ]:
def describe_table(table_name: str) -> list[tuple[str, str]]:
    """Look up the table schema.

    Returns:
      List of columns, where each entry is a tuple of (column, type).
    """
    print(' - DB CALL: describe_table')

    # Create a new connection for thread safety
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()

        cursor.execute(f"PRAGMA table_info({table_name});")

        schema = cursor.fetchall()
        # [column index, column name, column type, ...]
        return [(col[1], col[2]) for col in schema]


describe_table("products")

 - DB CALL: describe_table


[('product_id', 'INTEGER'),
 ('product_name', 'VARCHAR(255)'),
 ('price', 'DECIMAL(10, 2)')]

In [ ]:
def execute_query(sql: str) -> list[list[str]]:
    """Execute a SELECT statement, returning the results."""
    print(' - DB CALL: execute_query')

    # Create a new connection for thread safety
    with sqlite3.connect(db_file) as conn:
        cursor = conn.cursor()

        cursor.execute(sql)
        return cursor.fetchall()


execute_query("select * from products")

 - DB CALL: execute_query


[(1, 'Laptop', 799.99), (2, 'Keyboard', 129.99), (3, 'Mouse', 29.99)]

### Langkah 3: Bangun chatbot dengan create_react_agent

In [ ]:
# These are the Python functions defined above.
db_tools = [list_tables, describe_table, execute_query]

instruction = """You are a helpful chatbot that can interact with an SQL database for a computer
store. You will take the users questions and turn them into SQL queries using the tools
available. Once you have the information you need, you will answer the user's question using
the data returned. ALWAYS start by calling list_tables to discover the available tables.
ALWAYS call describe_table on the relevant table before writing any SQL query.
Never assume or guess table names or column names — always verify first."""

# Import the necessary modules from langgraph
from langchain.agents import create_agent

# Initialize the chat model with specific parameters
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0
)

# Create a ReAct agent using the model, tools, and prompt
agent = create_agent(
    model=model,
    tools=db_tools,
    system_prompt=instruction
)

perhatikan respons chatbot di bawah, LLM memanggil tool list_table -> describe_table -> execute_query untuk menganalisis jawaban dan memberikan jawaban yang tepat.

In [ ]:
# Run the agent
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Siapa saja orang yang terdaftar di database ini?"}]}
)

print(response["messages"][-1].content)

 - DB CALL: list_tables
 - DB CALL: describe_table
SELECT first_name, last_name FROM staff


In [ ]:
# Run the agent with a different query
response = agent.invoke(
    {"messages": [{"role": "user", "content": "Barang apa yang dijual di toko ini dan berapa harganya?"}]}
)

print(response["messages"][-1].content)

 - DB CALL: list_tables
 - DB CALL: describe_table


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 3.87986096s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerMod

# AI Agents with LangGraph Part 2

## Langkah 1: Define State dan Prompt

In [ ]:
system_instruction = """
You are a BaristaBot, an interactive cafe ordering system.
A human will talk to you about the available products you have and you will answer any questions about menu items
(and only about menu items - no off-topic discussion, but you can chat about the products and their history).
The customer will place an order for 1 or more items from the menu, which you will structure and send to the ordering system after confirming the order with the human.

Add items to the customer's order with add_to_order, and reset the order with clear_order.
To see the contents of the order so far, call get_order (this is shown to you, not the user)
Always confirm_order with the user (double-check) before calling place_order.
Calling confirm_order will display the order items to the user and returns their response to seeing the list.
Their response may contain modifications. Always verify and respond with drink and modifier names from the MENU before adding them to the order.
If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect.
You only have the modifiers listed on the menu.
Once the customer has finished ordering items, Call confirm_order to ensure it is correct then make any necessary updates and then call place_order.
Once place_order has returned, thank the user and say goodbye!
"""

## Langkah 2: Define Flow Chatbot dengan create_react_agent

In [ ]:
# In-memory database for the customer's order
customer_order = []

def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    # Note that this is just hard-coded text, but you could connect this to a live stock
    # database, or you could use Gemini's multi-modal capabilities and take live photos of
    # your cafe's chalk menu or the products on the counter and assmble them into an input.

    return """
    MENU:
    Coffee Drinks:
    Espresso
    Americano
    Cold Brew

    Coffee Drinks with Milk:
    Latte
    Cappuccino
    Cortado
    Macchiato
    Mocha
    Flat White

    Tea Drinks:
    English Breakfast Tea
    Green Tea
    Earl Grey

    Tea Drinks with Milk:
    Chai Latte
    Matcha Latte
    London Fog

    Other Drinks:
    Steamer
    Hot Chocolate

    Modifiers:
    Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default option: whole
    Espresso shots: Single, Double, Triple, Quadruple; default: Double
    Caffeine: Decaf, Regular; default: Regular
    Hot-Iced: Hot, Iced; Default: Hot
    Sweeteners (option to add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    Special requests: any reasonable modification that does not involve items not on the menu, for example: 'extra hot', 'one pump', 'half caff', 'extra foam', etc.

    "dirty" means add a shot of espresso to a drink that doesn't usually have it, like "Dirty Chai Latte".
    "Regular milk" is the same as 'whole milk'.
    "Sweetened" means add some regular sugar, not a sweetener.

    Soy milk has run out of stock today, so soy is not available.
    """

def add_to_order(item: str) -> str:
    """Add an item to the customer's order."""
    global customer_order
    customer_order.append(item)
    print(f"Adding '{item}' to order. Current order: {customer_order}")
    return f"I've added '{item}' to your order."

def clear_order() -> str:
    """Clear all items from the customer's order."""
    global customer_order
    customer_order.clear()
    print("Clearing order.")
    return "Your order has been cleared."

def get_order() -> list[str]:
    """Get the current items in the customer's order."""
    global customer_order
    print(f"Getting order. Current order: {customer_order}")
    return customer_order

def confirm_order() -> str:
    """Confirm the order with the customer."""
    global customer_order
    print(f"Confirming order. Current order: {customer_order}")
    if not customer_order:
        return "Your order is currently empty. What can I get for you?"

    order_string = ", ".join(customer_order)
    return f"Your order contains: {order_string}. Is this correct?"

def place_order() -> str:
    """Place the final order."""
    global customer_order
    print(f"Placing order. Final order: {customer_order}")
    if not customer_order:
        return "There's nothing in your order to place."

    final_order_summary = ", ".join(customer_order)
    # Clear the order after placing it, ready for the next customer
    customer_order.clear()
    return f"Your order for '{final_order_summary}' has been placed! It will be ready shortly."

# Collect all tools
barista_tools = [get_menu, add_to_order, clear_order, get_order, confirm_order, place_order]

# Import necessary modules
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

# Initialize the chat model
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.2
)

# Create a checkpointer for memory
checkpointer = InMemorySaver()

# Create the ReAct agent
barista_agent = create_agent(
    model=model,
    tools=barista_tools,
    system_prompt=system_instruction,
    checkpointer=checkpointer
)

### Interaksi dengan Barista Bot

In [ ]:
# Function to interact with the barista bot
def chat_with_barista():
    thread_id = "1"
    config = {"configurable": {"thread_id": thread_id}}

    print("Welcome to BaristaBot! Type 'q' to quit.")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ['q', 'quit', 'exit']:
            print("BaristaBot: Thank you for visiting! Have a great day!")
            break

        response = barista_agent.invoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": user_input
                    }
                ]
            },
            config=config
        )

        print("BaristaBot:", response["messages"][-1].content)

In [ ]:
# Run the interactive chat
# Uncomment the line below to start chatting with the barista bot
chat_with_barista()

Welcome to BaristaBot! Type 'q' to quit.
You: q
BaristaBot: Thank you for visiting! Have a great day!


### Contoh Penggunaan Langsung

In [ ]:
# Example of direct usage with specific queries
thread_id = "3"  # Unique identifier for this conversation
config = {"configurable": {"thread_id": thread_id}}

# User asks about menu
response = barista_agent.invoke(
    {"messages": [{"role": "user", "content": "What drinks do you have?"}]},
    config=config
)
print("User: What drinks do you have?")
print("BaristaBot:", response["messages"][-1].content)

In [ ]:
# User orders a drink
response = barista_agent.invoke(
    {"messages": [{"role": "user", "content": "I'd like a green tea please"}]},
    config=config
)
print("User: I'd like a green tea please")
print("BaristaBot:", response["messages"][-1].content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 8.873610216s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerMo

In [ ]:
# User orders a drink
response = barista_agent.invoke(
    {"messages": [{"role": "user", "content": "what is my order?"}]},
    config=config
)
print("User: what is my order?")
print("BaristaBot:", response["messages"][-1].content)

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 25.209159365s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerM

In [ ]:
!pip install -q streamlit pyngrok google-genai

In [ ]:
from pyngrok import ngrok
from google.colab import userdata

# Ambil token dari Colab Secrets
ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print("ngrok token berhasil dikonfigurasi!")

ngrok token berhasil dikonfigurasi!


In [ ]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    # Kill SEMUA proses streamlit, bukan hanya yang kita spawn
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

    # Force-free port kalau masih ada yang nempel
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)

    # Tutup semua tunnel ngrok
    ngrok.kill()

    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

In [ ]:
%%writefile streamlit_chat_app.py

import streamlit as st
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

st.title("Travel Umroh Agent")
st.caption("Temukan dan pesan paket Umroh impian Anda!")

# --- Umroh Agent Definitions ---
system_instruction = """
You are a Travel Umroh Agent, an interactive travel booking system.
A human will talk to you about available Umroh packages and you will answer any questions about destinations, prices, and itineraries.
The customer will select one or more packages, which you will structure and send to the booking system after confirming the order with the human.

Add items to the customer's booking with add_to_booking, and reset the booking with clear_booking.
To see the contents of the booking so far, call get_booking_details (this is shown to you, not the user).
Always confirm_booking with the user (double-check) before calling finalize_booking.
Calling confirm_booking will display the booking items to the user and returns their response to seeing the list.
Their response may contain modifications. Always verify and respond with package names and details from the MENU before adding them to the booking.
If you are unsure a package or detail matches those on the MENU, ask a question to clarify or redirect.
You only have the packages listed on the menu.
Once the customer has finished selecting packages, Call confirm_booking to ensure it is correct then make any necessary updates and then call finalize_booking.
Once finalize_booking has returned, thank the user and say goodbye!
"""

def get_umroh_packages() -> str:
    """Provides the latest up-to-date list of Umroh packages."""
    # This is hard-coded data, but could be connected to a live database.
    return """
    UMROH PACKAGES:

    Paket Hemat (10 Hari):
    - Harga: Rp 25.000.000
    - Maskapai: Saudi Airlines
    - Akomodasi: Hotel Bintang 3 (Makkah & Madinah)
    - Inklusi: Visa, Tiket Pesawat PP, Hotel, Makan 3x Sehari, Transportasi Lokal, Muthawwif Berpengalaman, Air Zamzam 5L.

    Paket Reguler (12 Hari):
    - Harga: Rp 30.000.000
    - Maskapai: Garuda Indonesia
    - Akomodasi: Hotel Bintang 4 (Makkah & Madinah)
    - Inklusi: Visa, Tiket Pesawat PP, Hotel, Makan 3x Sehari, Transportasi Lokal, Muthawwif Berpengalaman, City Tour, Air Zamzam 5L.

    Paket VIP (14 Hari):
    - Harga: Rp 45.000.000
    - Maskapai: Emirates/Qatar Airways
    - Akomodasi: Hotel Bintang 5 (Makkah & Madinah)
    - Inklusi: Visa, Tiket Pesawat PP (Direct Flight), Hotel Eksklusif, Makan 3x Sehari (Buffet Internasional), Transportasi Mewah, Muthawwif Private, City Tour Eksklusif, Perlengkapan Umroh Lengkap, Asuransi Perjalanan, Air Zamzam 5L.

    Untuk informasi lebih lanjut, silakan tanyakan detail paket yang Anda inginkan.
    """

def add_to_booking(package_name: str) -> str:
    """Adds an Umroh package to the customer's booking."""
    if "umroh_booking" not in st.session_state:
        st.session_state.umroh_booking = []
    st.session_state.umroh_booking.append(package_name)
    return f"Saya telah menambahkan paket '{package_name}' ke pemesanan Anda."

def clear_booking() -> str:
    """Clears all items from the customer's booking."""
    if "umroh_booking" in st.session_state:
        st.session_state.umroh_booking.clear()
    return "Pemesanan Anda telah dikosongkan."

def get_booking_details() -> list[str]:
    """Gets the current items in the customer's booking."""
    if "umroh_booking" not in st.session_state:
        st.session_state.umroh_booking = []
    return st.session_state.umroh_booking

def confirm_booking() -> str:
    """Confirms the booking with the customer."""
    if "umroh_booking" not in st.session_state:
        st.session_state.umroh_booking = []
    if not st.session_state.umroh_booking:
        return "Pemesanan Anda saat ini kosong. Ada paket Umroh yang bisa saya bantu?"

    booking_string = ", ".join(st.session_state.umroh_booking)
    return f"Pemesanan Anda berisi: {booking_string}. Apakah ini sudah benar?"

def finalize_booking() -> str:
    """Finalizes the customer's booking."""
    if "umroh_booking" not in st.session_state:
        st.session_state.umroh_booking = []
    if not st.session_state.umroh_booking:
        return "Tidak ada paket dalam pemesanan Anda untuk diselesaikan."

    final_booking_summary = ", ".join(st.session_state.umroh_booking)
    # Clear the booking after finalizing it, ready for the next customer
    st.session_state.umroh_booking.clear()
    return f"Pemesanan Anda untuk '{final_booking_summary}' telah berhasil! Anda akan segera dihubungi oleh tim kami. Terima kasih!"

umroh_agent_tools = [get_umroh_packages, add_to_booking, clear_booking, get_booking_details, confirm_booking, finalize_booking]
# --- End Umroh Agent Definitions ---

with st.sidebar:
    st.subheader("Pengaturan")
    google_api_key = st.text_input("Google AI API Key", type="password")
    reset_button = st.button("Reset Percakapan", help="Hapus semua pesan dan mulai dari awal")

if not google_api_key:
    st.info("Masukkan Google AI API Key di sidebar untuk mulai chat.", icon="🗝️")
    st.stop()

# --- Inisialisasi Umroh Agent ---
if ("umroh_agent" not in st.session_state) or (getattr(st.session_state, "_last_key", None) != google_api_key):
    try:
        os.environ["GOOGLE_API_KEY"] = google_api_key
        st.session_state.model = ChatGoogleGenerativeAI(
            model="gemini-2.5-flash-lite",
            temperature=0.2
        )
        st.session_state.checkpointer = InMemorySaver()

        st.session_state.umroh_agent = create_agent(
            model=st.session_state.model,
            tools=umroh_agent_tools,
            system_prompt=system_instruction,
            checkpointer=st.session_state.checkpointer
        )

        st.session_state._last_key = google_api_key

        # Clear session state for new agent/key
        st.session_state.pop("messages", None)
        st.session_state.pop("umroh_booking", None)
        if "thread_id" in st.session_state:
            del st.session_state["thread_id"]
        st.session_state.thread_id = "streamlit_umroh_chat" # Fixed thread ID for Streamlit app

    except Exception as e:
        st.error(f"API Key tidak valid atau error inisialisasi agent: {e}")
        st.stop()

# Inisialisasi thread_id dan umroh_booking
if "thread_id" not in st.session_state:
    st.session_state.thread_id = "streamlit_umroh_chat"
if "umroh_booking" not in st.session_state:
    st.session_state.umroh_booking = []

# --- Tombol Reset ---
if reset_button:
    st.session_state.pop("messages", None)
    st.session_state.pop("umroh_booking", None)
    if "thread_id" in st.session_state:
        del st.session_state["thread_id"]
    st.rerun()

# --- Tampilkan Riwayat Percakapan ---
if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# --- Input & Respons ---
prompt = st.chat_input("Ketik pesanmu di sini...")

if prompt:
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    try:
        config = {"configurable": {"thread_id": st.session_state.thread_id}}
        response_agent = st.session_state.umroh_agent.invoke(
            {"messages": [{"role": "user", "content": prompt}]},
            config=config
        )

        if "messages" in response_agent and len(response_agent["messages"]) > 0:
            answer = response_agent["messages"][-1].content
        else:
            answer = "Maaf, saya tidak bisa memproses permintaan Anda."

    except Exception as e:
        answer = f"Terjadi error saat memanggil agen: {e}"

    with st.chat_message("assistant"):
        st.markdown(answer)

    st.session_state.messages.append({"role": "assistant", "content": answer})


Overwriting streamlit_chat_app.py


In [ ]:
proc = run_streamlit("streamlit_chat_app.py")

Streamlit berjalan: NgrokTunnel: "https://royal-overbid-stem.ngrok-free.dev" -> "http://localhost:8501"
